In [1]:
# ==============================================================================
# Step 1: Setup and Imports
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import pandas as pd

# Check if a GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==============================================================================
# Step 2: Custom Dataset Using CSV
# ==============================================================================
class JewelryCSVDataset(Dataset):
    """Custom Dataset for loading jewelry images with labels from CSV."""
    def __init__(self, csv_file, img_dir, transform=None):
        """
        Args:
            csv_file (str): Path to CSV file with annotations.
            img_dir (str): Root directory containing all images.
            transform (callable, optional): Optional transform to apply on images.
        """
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        # Map jewelry_type to numeric labels
        self.class_names = sorted(self.data['jewelry_type'].unique())
        self.class_to_idx = {name: i for i, name in enumerate(self.class_names)}
        self.data['label'] = self.data['jewelry_type'].map(self.class_to_idx)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Get image path from CSV
        img_path = os.path.join(self.img_dir, self.data.iloc[idx]['image_path'])
        image = Image.open(img_path).convert('RGB')

        label = self.data.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, label


# Define transformations for the images
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# ==============================================================================
# Step 3: Load Dataset and Create DataLoaders
# ==============================================================================
# Example paths (update these to your actual Kaggle input paths)
CSV_PATH = "/kaggle/input/jewelry-images/augmented_dataset.csv"   # <-- your CSV file
IMG_DIR = "/kaggle/input/jewelry-images/augment_images/content/Augment"  # <-- your images folder

# Full dataset
full_dataset = JewelryCSVDataset(csv_file=CSV_PATH, img_dir=IMG_DIR, transform=data_transforms)

# Split into train and validation sets (e.g., 80/20 split)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Get class names
class_names = full_dataset.class_names
print(f"Classes found: {class_names}")


# ==============================================================================
# Step 4: Define Model (Unchanged)
# ==============================================================================
class VGG16_GRU_Classifier(nn.Module):
    def __init__(self, num_classes, hidden_size=256, gru_layers=1):
        super(VGG16_GRU_Classifier, self).__init__()
        vgg16 = models.vgg16(pretrained=True)
        self.features = vgg16.features

        for param in self.features.parameters():
            param.requires_grad = False

        self.gru = nn.GRU(input_size=512, hidden_size=hidden_size,
                          num_layers=gru_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.features(x)
        batch_size, num_channels, height, width = x.size()
        x = x.view(batch_size, num_channels, height * width)
        x = x.permute(0, 2, 1)  # (batch_size, 49, 512)
        _, h_n = self.gru(x)
        h_n = h_n.squeeze(0)
        out = self.fc(h_n)
        return out


# Initialize the model
num_classes = len(class_names)
model = VGG16_GRU_Classifier(num_classes=num_classes).to(device)


# ==============================================================================
# Step 5: Training Loop
# ==============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Validation Accuracy: {accuracy:.2f}%')

print("Finished Training!")


# ==============================================================================
# Step 6: Prediction Function
# ==============================================================================
def predict_image(image_path, model, class_names):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    transformed_image = data_transforms(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(transformed_image)
        _, predicted_idx = torch.max(output, 1)
    return class_names[predicted_idx.item()]


Using device: cuda
Classes found: ['earring', 'necklace']


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 228MB/s] 
/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/10], Loss: 0.1875, Validation Accuracy: 93.97%
Epoch [2/10], Loss: 0.1115, Validation Accuracy: 94.82%
Epoch [3/10], Loss: 0.0764, Validation Accuracy: 94.85%
Epoch [4/10], Loss: 0.0600, Validation Accuracy: 94.87%
Epoch [5/10], Loss: 0.0485, Validation Accuracy: 95.15%
Epoch [6/10], Loss: 0.0422, Validation Accuracy: 95.13%
Epoch [7/10], Loss: 0.0341, Validation Accuracy: 95.21%
Epoch [8/10], Loss: 0.0315, Validation Accuracy: 94.96%
Epoch [9/10], Loss: 0.0291, Validation Accuracy: 94.61%
Epoch [10/10], Loss: 0.0270, Validation Accuracy: 94.91%
Finished Training!


In [2]:
# Save the entire model (architecture + weights)
torch.save(model, "jewelry_model_full.pth")
print("✅ Full model saved as jewelry_model_full.pth")

✅ Full model saved as jewelry_model_full.pth


In [3]:
# Save weights only (recommended way)
torch.save(model.state_dict(), "jewelry_model.pth")

# Recreate the model before loading
loaded_model = VGG16_GRU_Classifier(num_classes=len(class_names)).to(device)
loaded_model.load_state_dict(torch.load("jewelry_model.pth", map_location=device))
loaded_model.eval()
print("✅ Model loaded (weights only)")

✅ Model loaded (weights only)


In [5]:
print("\nPrediction function is ready. Use it like this:")
print("predicted_type = ", predict_image('/kaggle/input/img2-jpg/img2.webp', model, class_names))


Prediction function is ready. Use it like this:
predicted_type =  earring


In [6]:
# Converts a PIL image to prediction using your trained model
def predict_image_from_PIL(image, model, class_names, transform, device):
    model.eval()
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted_idx = torch.max(output, 1)
    return class_names[predicted_idx.item()]

# Wrapper function that UI will call
def my_predict_function(image):
    return predict_image_from_PIL(image, model, class_names, data_transforms, device)

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import io

def create_jewelry_ui(predict_function):
    """
    predict_function: function that takes a PIL Image and returns predicted class (string)
    """
    # Accept common image formats
    upload = widgets.FileUpload(accept='.jpg,.jpeg,.png,.webp,.avif', multiple=False)
    predict_btn = widgets.Button(description='Predict Jewelry Type', button_style='success')
    output = widgets.Output()

    def on_predict(_):
        with output:
            clear_output()
            if not upload.value:
                print("⚠️ Please upload an image first")
                return

            try:
                # Handle both Kaggle and Colab formats
                if isinstance(upload.value, dict):
                    file_name = next(iter(upload.value))
                    img_bytes = upload.value[file_name]['content']
                elif isinstance(upload.value, (tuple, list)):
                    img_bytes = upload.value[0].content
                else:
                    print("❌ Unable to read uploaded file. Unsupported format.")
                    return

                # Load image (PIL supports jpg, jpeg, png, webp, avif)
                image = Image.open(io.BytesIO(img_bytes)).convert('RGB')

            except Exception as e:
                print(f"❌ Unable to read uploaded file. Error: {e}")
                return

            # Predict
            pred_class = predict_function(image)

            # Display image + prediction
            plt.figure(figsize=(6,6))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"💎 Predicted Jewelry Type: {pred_class}", fontsize=14, pad=15)
            plt.show()

    predict_btn.on_click(on_predict)

    # Display UI
    display(widgets.HTML("<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>"))
    display(upload)
    display(predict_btn)
    display(output)


# ======================
# USAGE
# ======================
# my_predict_function = wrapper around your model/predict_image function
create_jewelry_ui(my_predict_function)

HTML(value='<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>')

FileUpload(value=(), accept='.jpg,.jpeg,.png,.webp,.avif', description='Upload')

Button(button_style='success', description='Predict Jewelry Type', style=ButtonStyle())

Output()

In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import io

def create_jewelry_ui(predict_function):
    """
    predict_function: function that takes a PIL Image and returns predicted class (string)
    """
    # Accept common image formats
    upload = widgets.FileUpload(accept='.jpg,.jpeg,.png,.webp,.avif', multiple=False)
    predict_btn = widgets.Button(description='Predict Jewelry Type', button_style='success')
    output = widgets.Output()

    def on_predict(_):
        with output:
            clear_output()
            if not upload.value:
                print("⚠️ Please upload an image first")
                return

            try:
                # Handle both Kaggle and Colab formats
                if isinstance(upload.value, dict):
                    file_name = next(iter(upload.value))
                    img_bytes = upload.value[file_name]['content']
                elif isinstance(upload.value, (tuple, list)):
                    img_bytes = upload.value[0].content
                else:
                    print("❌ Unable to read uploaded file. Unsupported format.")
                    return

                # Load image (PIL supports jpg, jpeg, png, webp, avif)
                image = Image.open(io.BytesIO(img_bytes)).convert('RGB')

            except Exception as e:
                print(f"❌ Unable to read uploaded file. Error: {e}")
                return

            # Predict
            pred_class = predict_function(image)

            # Display image + prediction
            plt.figure(figsize=(6,6))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"💎 Predicted Jewelry Type: {pred_class}", fontsize=14, pad=15)
            plt.show()

    predict_btn.on_click(on_predict)

    # Display UI
    display(widgets.HTML("<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>"))
    display(upload)
    display(predict_btn)
    display(output)


# ======================
# USAGE
# ======================
# my_predict_function = wrapper around your model/predict_image function
create_jewelry_ui(my_predict_function)

HTML(value='<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>')

FileUpload(value=(), accept='.jpg,.jpeg,.png,.webp,.avif', description='Upload')

Button(button_style='success', description='Predict Jewelry Type', style=ButtonStyle())

Output()

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import io

def create_jewelry_ui(predict_function):
    """
    predict_function: function that takes a PIL Image and returns predicted class (string)
    """
    # Accept common image formats
    upload = widgets.FileUpload(accept='.jpg,.jpeg,.png,.webp,.avif', multiple=False)
    predict_btn = widgets.Button(description='Predict Jewelry Type', button_style='success')
    output = widgets.Output()

    def on_predict(_):
        with output:
            clear_output()
            if not upload.value:
                print("⚠️ Please upload an image first")
                return

            try:
                # Handle both Kaggle and Colab formats
                if isinstance(upload.value, dict):
                    file_name = next(iter(upload.value))
                    img_bytes = upload.value[file_name]['content']
                elif isinstance(upload.value, (tuple, list)):
                    img_bytes = upload.value[0].content
                else:
                    print("❌ Unable to read uploaded file. Unsupported format.")
                    return

                # Load image (PIL supports jpg, jpeg, png, webp, avif)
                image = Image.open(io.BytesIO(img_bytes)).convert('RGB')

            except Exception as e:
                print(f"❌ Unable to read uploaded file. Error: {e}")
                return

            # Predict
            pred_class = predict_function(image)

            # Display image + prediction
            plt.figure(figsize=(6,6))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"💎 Predicted Jewelry Type: {pred_class}", fontsize=14, pad=15)
            plt.show()

    predict_btn.on_click(on_predict)

    # Display UI
    display(widgets.HTML("<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>"))
    display(upload)
    display(predict_btn)
    display(output)


# ======================
# USAGE
# ======================
# my_predict_function = wrapper around your model/predict_image function
create_jewelry_ui(my_predict_function)

HTML(value='<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>')

FileUpload(value=(), accept='.jpg,.jpeg,.png,.webp,.avif', description='Upload')

Button(button_style='success', description='Predict Jewelry Type', style=ButtonStyle())

Output()

In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import io

def create_jewelry_ui(predict_function):
    """
    predict_function: function that takes a PIL Image and returns predicted class (string)
    """
    # Accept common image formats
    upload = widgets.FileUpload(accept='.jpg,.jpeg,.png,.webp,.avif', multiple=False)
    predict_btn = widgets.Button(description='Predict Jewelry Type', button_style='success')
    output = widgets.Output()

    def on_predict(_):
        with output:
            clear_output()
            if not upload.value:
                print("⚠️ Please upload an image first")
                return

            try:
                # Handle both Kaggle and Colab formats
                if isinstance(upload.value, dict):
                    file_name = next(iter(upload.value))
                    img_bytes = upload.value[file_name]['content']
                elif isinstance(upload.value, (tuple, list)):
                    img_bytes = upload.value[0].content
                else:
                    print("❌ Unable to read uploaded file. Unsupported format.")
                    return

                # Load image (PIL supports jpg, jpeg, png, webp, avif)
                image = Image.open(io.BytesIO(img_bytes)).convert('RGB')

            except Exception as e:
                print(f"❌ Unable to read uploaded file. Error: {e}")
                return

            # Predict
            pred_class = predict_function(image)

            # Display image + prediction
            plt.figure(figsize=(6,6))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"💎 Predicted Jewelry Type: {pred_class}", fontsize=14, pad=15)
            plt.show()

    predict_btn.on_click(on_predict)

    # Display UI
    display(widgets.HTML("<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>"))
    display(upload)
    display(predict_btn)
    display(output)


# ======================
# USAGE
# ======================
# my_predict_function = wrapper around your model/predict_image function
create_jewelry_ui(my_predict_function)

HTML(value='<h3>💎 Jewelry Classifier</h3><p>Upload an image of jewelry to predict its type.</p>')

FileUpload(value=(), accept='.jpg,.jpeg,.png,.webp,.avif', description='Upload')

Button(button_style='success', description='Predict Jewelry Type', style=ButtonStyle())

Output()